# 22_Image 모델에 맞게 데이터 변형하기

## 학습목표 
- 1. 이미지와 라벨을 추출하는 CustomDataset 클래스를 활용하여 알맞은 형태의 DataLoder를 제작합니다. 
- 2. 데이터 변형에서 중요한 의미를 갖는 옵션들을 살펴봅니다.

In [1]:
import os #로컬 컴퓨터에서 데이터셋을 불러오기 위해
import cv2 # 시각화 해서 표현하기 위함
import numpy as np #이미지 픽셀 데이터를 numpy 형태로 표현함
import json #개별 라벨을 읽어들어오게 함

import torch #훈련
from torch.utils.data import Dataset, DataLoader #커스텀 데이터셋을 만들기 위함
from torchvision import transforms #torchvision -> 딥러닝(이미지) 수행하는 클래스 이름 #transform 이미지를 조절
from PIL import Image #-> 이미지를 표현할 수 있는 파이썬 라이브러리 
import matplotlib.pyplot as plt

In [ ]:
# 타겟 딕셔너리 생성
target = {
    "boxes": boxes,
    "labels": labels,
    "image_id": image_id}

In [ ]:
b = [[241, 386, 480, 606], [82, 744, 384, 1059], [1347, 722, 1649, 1045]]
print(len(b))

3


In [ ]:
class CustomData(Dataset):
    def __init__(self, image_list, label_list, transform):
        self.image_path = self.read_file_lines(image_list)
        self.label_path = self.read_file_lines(label_list)
        self.transform = transform
        
    def __len__(self):
        #if (이미지수 == 라벨수) return 이미지 수
        return len(self.image_path)

    #텐서 형태로 변환된 이미지를 돌려줌
    def __getitem__(self, idx):
        image = self.image_path[idx]
        image = Image.open(image)    #해당 경로에서 이미지를 읽어 옴

        if self.transform is not None:
            result = self.transform(image)

        box = self.get_label_data(self.label_path[idx])

        #타겟 딕셔너리 생성
        target = {
            'boxes' : box,
            #박스의 갯수만큼 라벨을 출력해주어야 함. 우리는 클래스는 하나이므로 그냥 활용해도 됨.
            'labels' : torch.zeros(len(box), dtype=torch.int64),
            'image_id' : torch.tensor([idx], dtype=torch.int64)
        }
        #print(f'box:{box}, labels:{torch.zeros(len(box), dtype=torch.int64)}, image_id : {torch.tensor([idx], dtype=torch.int64)}')
        
        return image, target

    def get_label_data(self, lab):
        bounding_boxes = []
        with open(lab, 'r') as f:
            annotations = json.load(f)
            
        for i in range(len(annotations['tooth'])):
            if annotations['tooth'][i]['decayed'] == True:
                # x, y 좌표를 각각 추출
                # p[0] 기준으로 오름차순 정렬
                sorted_xs = sorted(annotations['tooth'][i]['segmentation'], key=lambda p: p[0])
                
                start_x = sorted_xs[0][0] 
                end_x = sorted_xs[-1][0]
                
                sorted_ys = sorted(annotations['tooth'][i]['segmentation'], key=lambda p: p[1])
                start_y = sorted_ys[0][1] 
                end_y = sorted_ys[-1][1]

                #x_min, y_min, x_max, y_max
                bounding_boxes.append([start_x, start_y, end_x, end_y])

        return bounding_boxes

    def read_file_lines(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f]  # 개행 문자 제거 후 리스트로 변환
        return lines

In [ ]:
from torchvision import transforms #torchvision -> 딥러닝(이미지) 수행하는 클래스 이름 #transform 이미지를 조절
from PIL import Image #-> 이미지를 표현할 수 있는 파이썬 라이브러리 

In [ ]:
transform = transforms.Compose([
    #전처리의 다양한 종류를 여기서 적용해준다.
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### 이미지 데이터를 가져와서 로드함

In [ ]:
label_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/train_list.txt'
image_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/train_image_list.txt'

In [ ]:
def custom_collate_fn(batch):
    images = []
    targets = []

    for image, target in batch:
        images.append(image)  # 이미지 리스트
        targets.append(target)  # 딕셔너리 형태의 target 리스트

    return images, targets  # 리스트 형태 그대로 반환

In [ ]:
dataset = CustomData(image_list, label_list, transform) #이미지에 대한 경로, 일괄적으로 적용할 전처리
train_dataloader = DataLoader(dataset = dataset,
                        batch_size = 25,
                        shuffle = True,
                        drop_last = False,
                        collate_fn=custom_collate_fn)

In [ ]:
dataiter = iter(train_dataloader)
batch = next(dataiter)

images, labels = batch

box:[[269, 389, 502, 598], [189, 550, 422, 738], [71, 700, 357, 1005], [1232, 755, 1509, 1060]], labels:tensor([0, 0, 0, 0]), image_id : tensor([10])
box:[[153, 496, 429, 731], [81, 714, 408, 1063], [1320, 658, 1656, 1026]], labels:tensor([0, 0, 0]), image_id : tensor([13])
box:[[280, 301, 556, 533], [84, 659, 412, 990], [1257, 325, 1528, 552], [1360, 702, 1696, 1058]], labels:tensor([0, 0, 0, 0]), image_id : tensor([16])
box:[[320, 239, 558, 491], [214, 465, 471, 712], [74, 643, 422, 1017], [947, 27, 1214, 264], [1345, 248, 1587, 496], [1517, 659, 1838, 1005]], labels:tensor([0, 0, 0, 0, 0, 0]), image_id : tensor([7])
box:[[732, 26, 995, 265], [530, 79, 757, 294], [347, 178, 585, 428], [93, 575, 430, 919], [990, 37, 1248, 289], [1211, 119, 1441, 345], [1383, 235, 1625, 482], [1561, 655, 1832, 1010]], labels:tensor([0, 0, 0, 0, 0, 0, 0, 0]), image_id : tensor([9])
box:[[668, 26, 896, 229], [497, 78, 683, 250], [236, 419, 470, 632], [231, 619, 445, 815], [89, 752, 389, 1056], [1241, 135